# FLUX Runtime Re-Embed — Fix Import Issues

**Purpose:** Re-embed the fixed Python runtime files into Flux-Apex-V1.flx

**What Was Fixed:**
- Changed relative imports to try absolute imports first
- Pattern: `from module import X` → `try: from phases.X import ... except: from .X import ...`

**Files Fixed:**
- phases/phase1/cse.py
- phases/phase2/field.py
- phases/phase3/gravity.py
- phases/phase4/*.py
- phases/phase5/*.py
- phases/phase6/*.py
- phases/phase8/flux_large.py
- phases/phase8_8/grid_adapters.py
- phases/phase8_9/*.py
- phases/phase_orchestrator/*.py
- phases/phase_unified/unified_agent.py
- phases/phase_claw/*.py

In [ ]:
# Cell 1: Setup
import sys
import os
import gc
import gzip
import base64
import torch
from pathlib import Path
from datetime import datetime

# Environment detection
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
ON_COLAB = 'COLAB_GPU' in os.environ

if ON_KAGGLE:
    !git clone https://github.com/Unseengap/FLUX.git /kaggle/working/flux 2>/dev/null || git -C /kaggle/working/flux pull
    ROOT = Path('/kaggle/working/flux')
elif ON_COLAB:
    !git clone https://github.com/Unseengap/FLUX.git /content/flux 2>/dev/null || git -C /content/flux pull
    ROOT = Path('/content/flux')
else:
    ROOT = Path('.').resolve()
    if not (ROOT / 'flux_model.py').exists():
        ROOT = ROOT.parent

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

MODEL_PATH = ROOT / 'checkpoints' / 'Flux-Apex-V1.flx'

print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Colab' if ON_COLAB else 'Local'}")
print(f"Root: {ROOT}")
print(f"Model: {MODEL_PATH}")

In [ ]:
# Cell 2: Download Model if needed
from huggingface_hub import hf_hub_download

MODEL_PATH.parent.mkdir(exist_ok=True)

if not MODEL_PATH.exists():
    HF_TOKEN = None
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        except: pass
    
    print("📥 Downloading Flux-Apex-V1.flx...")
    hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-Apex-V1.flx',
        local_dir=str(ROOT),
        token=HF_TOKEN,
    )
    print(f"✓ Downloaded")
else:
    print(f"✓ Model exists: {MODEL_PATH.stat().st_size / 1e9:.2f} GB")

In [ ]:
# Cell 3: Define file collection tiers

# Files to embed (Tier 1-3 from FLUX_CODEBASE_EMBED_SPEC.md)
TIER_1_CRITICAL = [
    # Root level
    'flux_model.py',
    'flux_utils.py',
    'flux_inspector.py',
    'flux_lazy_loader.py',
    'bootstrap.py',
    
    # Phase 1: CSE
    'phases/phase1/__init__.py',
    'phases/phase1/cse.py',
    'phases/phase1/wave_types.py',
    'phases/phase1/interference.py',
    
    # Phase 2: Field
    'phases/phase2/__init__.py',
    'phases/phase2/field.py',
    'phases/phase2/flux_format.py',
    'phases/phase2/attractor.py',
    'phases/phase2/field_ops.py',
    
    # Phase 3: Gravity
    'phases/phase3/__init__.py',
    'phases/phase3/gravity.py',
    'phases/phase3/mass_tracker.py',
    'phases/phase3/spatial_index.py',
    'phases/phase3/negative_mass.py',
    
    # Phase 4: Thermodynamic
    'phases/phase4/__init__.py',
    'phases/phase4/thermodynamic.py',
    'phases/phase4/temperature.py',
    'phases/phase4/energy_functions.py',
    'phases/phase4/online_learner.py',
    
    # Phase 5: CGN
    'phases/phase5/__init__.py',
    'phases/phase5/cgn.py',
    'phases/phase5/causal_graph.py',
    'phases/phase5/manifold.py',
    'phases/phase5/multi_timescale.py',
    
    # Phase 6: Memory
    'phases/phase6/__init__.py',
    'phases/phase6/working_memory.py',
    'phases/phase6/episodic_memory.py',
    'phases/phase6/semantic_memory.py',
    'phases/phase6/memory_router.py',
    'phases/phase6/consolidation.py',
    
    # Phase 8: Decoder (legacy but needed)
    'phases/phase8/__init__.py',
    'phases/phase8/wave_decoder.py',
    'phases/phase8/flux_large.py',
    
    # Phase 8.8: Grid adapters
    'phases/phase8_8/__init__.py',
    'phases/phase8_8/grid_adapters.py',
    'phases/phase8_8/flx_loader.py',
    'phases/phase8_8/wave_to_x.py',
]

TIER_2_ORCHESTRATION = [
    # Orchestrator
    'phases/phase_orchestrator/__init__.py',
    'phases/phase_orchestrator/flux_orchestrator.py',
    'phases/phase_orchestrator/tool_registry.py',
    'phases/phase_orchestrator/system_prompt.py',
    'phases/phase_orchestrator/embed_orchestration.py',
    
    # Spatial Memory (ARC)
    'phases/phase9_arc/__init__.py',
    'phases/phase9_arc/spatial_memory.py',
    
    # Unified Agent
    'phases/phase_unified/__init__.py',
    'phases/phase_unified/unified_agent.py',
    'phases/phase_unified/frame_differ.py',
    'phases/phase_unified/strategies.py',
    'phases/phase_unified/rendering.py',
    'phases/phase_unified/goal_planner.py',
    'phases/phase_unified/causal_tracker.py',
    'phases/phase_unified/rule_inducer.py',
    'phases/phase_unified/perception_field.py',
    'phases/phase_unified/arc_session.py',
    'phases/phase_unified/arc_interface.py',
    'phases/phase_unified/game_loop.py',
]

TIER_3_MULTIMODAL = [
    # Phase 8.9: Multi-modal
    'phases/phase8_9/__init__.py',
    'phases/phase8_9/image_adapters.py',
    'phases/phase8_9/audio_adapters.py',
    'phases/phase8_9/flux_to_any.py',
    
    # Phase 9 ARC extras
    'phases/phase9_arc/pattern_library.py',
    'phases/phase9_arc/rule_inducer.py',
    'phases/phase9_arc/arc_agent.py',
    'phases/phase9_arc/object_detector.py',
    'phases/phase9_arc/arc_loader.py',
]

TIER_4_CLAW = [
    # CLAW harness
    'phases/phase_claw/__init__.py',
    'phases/phase_claw/flux_bridge.py',
    'phases/phase_claw/runtime.py',
    'phases/phase_claw/tools.py',
    'phases/phase_claw/commands.py',
    'phases/phase_claw/query_engine.py',
    'phases/phase_claw/tool_pool.py',
    'phases/phase_claw/models.py',
    'phases/phase_claw/permissions.py',
    'phases/phase_claw/context.py',
    'phases/phase_claw/session_store.py',
    'phases/phase_claw/history.py',
    'phases/phase_claw/transcript.py',
    'phases/phase_claw/system_init.py',
    'phases/phase_claw/port_manifest.py',
    'phases/phase_claw/parity_audit.py',
    'phases/phase_claw/setup.py',
    'phases/phase_claw/execution_registry.py',
    # Additional fixed files
    'phases/phase_claw/main.py',
    'phases/phase_claw/command_graph.py',
    'phases/phase_claw/task.py',
    'phases/phase_claw/tasks.py',
    'phases/phase_claw/QueryEngine.py',
    'phases/phase_claw/costHook.py',
    # Missing dependencies (stubs)
    'phases/phase_claw/deferred_init.py',
    'phases/phase_claw/bootstrap_graph.py',
    'phases/phase_claw/cost_tracker.py',
    'phases/phase_claw/prefetch.py',
    'phases/phase_claw/direct_modes.py',
    'phases/phase_claw/remote_runtime.py',
]

ALL_FILES = TIER_1_CRITICAL + TIER_2_ORCHESTRATION + TIER_3_MULTIMODAL + TIER_4_CLAW

print(f"Total files to embed: {len(ALL_FILES)}")
print(f"  Tier 1 (Critical): {len(TIER_1_CRITICAL)}")
print(f"  Tier 2 (Orchestration): {len(TIER_2_ORCHESTRATION)}")
print(f"  Tier 3 (Multi-modal): {len(TIER_3_MULTIMODAL)}")
print(f"  Tier 4 (CLAW): {len(TIER_4_CLAW)}")

In [ ]:
# Cell 4: Collect and validate files

def collect_files(file_list: list) -> dict:
    """Collect file contents, validate syntax."""
    collected = {}
    errors = []
    
    for file_path in file_list:
        full_path = ROOT / file_path
        
        if not full_path.exists():
            print(f"⚠ Missing: {file_path}")
            continue
            
        source = full_path.read_text(encoding='utf-8')
        
        # Validate syntax
        try:
            compile(source, file_path, 'exec')
            collected[file_path] = source
            print(f"✓ {file_path} ({len(source)} chars)")
        except SyntaxError as e:
            errors.append((file_path, str(e)))
            print(f"✗ {file_path} - SYNTAX ERROR: {e}")
    
    return collected, errors

print("Collecting files...\n")
code_bundle, syntax_errors = collect_files(ALL_FILES)

print(f"\n✓ Collected {len(code_bundle)} files")
if syntax_errors:
    print(f"✗ {len(syntax_errors)} syntax errors - MUST FIX before embedding")

In [ ]:
# Cell 5: Compress code bundle

def compress_source(source: str) -> str:
    """Compress source code using gzip + base64."""
    compressed = gzip.compress(source.encode('utf-8'))
    return base64.b64encode(compressed).decode('ascii')

# Compute statistics
total_lines = sum(src.count('\n') + 1 for src in code_bundle.values())
raw_size = sum(len(src.encode('utf-8')) for src in code_bundle.values())

# Compress
compressed_bundle = {path: compress_source(src) for path, src in code_bundle.items()}
compressed_size = sum(len(c) for c in compressed_bundle.values())

print(f"\nCode Bundle Statistics:")
print(f"  Files: {len(code_bundle)}")
print(f"  Lines: {total_lines:,}")
print(f"  Raw size: {raw_size:,} bytes ({raw_size/1024:.1f} KB)")
print(f"  Compressed: {compressed_size:,} bytes ({compressed_size/1024:.1f} KB)")
print(f"  Ratio: {raw_size/compressed_size:.2f}x")

In [ ]:
# Cell 6: Load current model

print(f"Loading {MODEL_PATH}...")
flx = torch.load(str(MODEL_PATH), map_location='cpu', weights_only=False)

print(f"\nCurrent model:")
print(f"  Version: {flx.get('version', 'unknown')}")
print(f"  Format: {flx.get('format', 'unknown')}")

if 'runtime' in flx:
    old_runtime = flx['runtime']
    old_files = len(old_runtime.get('code', {}))
    old_version = old_runtime.get('version', 'unknown')
    print(f"  Old runtime: {old_version} ({old_files} files)")
else:
    print(f"  No existing runtime")

In [ ]:
# Cell 7: Update runtime

# Create new runtime section
new_runtime = {
    'version': '8.2-fixed-imports',
    'embedded_at': datetime.now().isoformat(),
    'bootstrap': (ROOT / 'bootstrap.py').read_text(),
    'code': compressed_bundle,
    'metadata': {
        'total_files': len(code_bundle),
        'total_lines': total_lines,
        'raw_size_bytes': raw_size,
        'compressed_size_bytes': compressed_size,
        'compression_ratio': raw_size / compressed_size,
        'tiers': {
            'tier1_critical': len(TIER_1_CRITICAL),
            'tier2_orchestration': len(TIER_2_ORCHESTRATION),
            'tier3_multimodal': len(TIER_3_MULTIMODAL),
            'tier4_claw': len(TIER_4_CLAW),
        },
        'fixes': [
            'Changed relative imports to try absolute imports first',
            'Pattern: try: from phases.X import ... except: from .X import ...',
            'Files affected: cse.py, field.py, gravity.py, thermodynamic.py, cgn.py, memory_router.py, etc.',
        ],
    },
}

# Update model
flx['runtime'] = new_runtime
flx['version'] = '8.2-fixed-imports'
flx['metadata']['last_modified'] = datetime.now().isoformat()
flx['metadata']['modified_components'] = flx['metadata'].get('modified_components', []) + ['runtime']

print(f"✓ Runtime updated")
print(f"  New version: {new_runtime['version']}")
print(f"  Files: {len(code_bundle)}")
print(f"  Lines: {total_lines:,}")

In [ ]:
# Cell 8: Save updated model

# On disk-constrained environments, delete original first
import shutil

_, _, disk_free = shutil.disk_usage(MODEL_PATH.parent)
disk_free_gb = disk_free / 1e9
model_size_gb = MODEL_PATH.stat().st_size / 1e9 if MODEL_PATH.exists() else 0

print(f"Disk free: {disk_free_gb:.1f} GB")
print(f"Model size: {model_size_gb:.2f} GB")

# If low on disk, delete original first (model is in memory)
if disk_free_gb < model_size_gb + 5:
    print("\n⚠ Low disk space - deleting original before save...")
    if MODEL_PATH.exists():
        os.remove(MODEL_PATH)
        gc.collect()
        print(f"✓ Freed {model_size_gb:.2f} GB")

# Save
print(f"\nSaving to {MODEL_PATH}...")
torch.save(flx, str(MODEL_PATH))

new_size = MODEL_PATH.stat().st_size / 1e9
print(f"✓ Saved: {new_size:.2f} GB")

In [ ]:
# Cell 9: Verify by running bootstrap

print("\nVerifying bootstrap works...\n")

# Clear any cached modules
for key in list(sys.modules.keys()):
    if key.startswith('phases.'):
        del sys.modules[key]

# Import and test bootstrap
from bootstrap import wake_up

result = wake_up(str(MODEL_PATH), device='cpu', verbose=True)

print(f"\n{'='*60}")
print("BOOTSTRAP VERIFICATION")
print(f"{'='*60}")
print(f"Modules loaded: {len(result.get('modules', {}))}")
print(f"FLX loaded: {'Yes' if result.get('flx') else 'No'}")

# Count errors vs successes
# (will be visible from the wake_up output above)

In [ ]:
# Cell 10: Upload to HuggingFace (optional)

UPLOAD_TO_HF = False  # Set to True to upload

if UPLOAD_TO_HF:
    from huggingface_hub import HfApi
    
    HF_TOKEN = None
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        except: pass
    else:
        HF_TOKEN = os.environ.get('HF_TOKEN')
    
    if HF_TOKEN:
        api = HfApi()
        print("Uploading to HuggingFace...")
        api.upload_file(
            path_or_fileobj=str(MODEL_PATH),
            path_in_repo='checkpoints/Flux-Apex-V1.flx',
            repo_id='UnseenGAP/FLUX',
            token=HF_TOKEN,
            commit_message='Fix embedded imports for bootstrap compatibility',
        )
        print("✓ Uploaded")
    else:
        print("⚠ No HF_TOKEN - cannot upload")
else:
    print("Upload disabled. Set UPLOAD_TO_HF = True to upload.")

In [ ]:
# Cell 11: Summary

print(f"\n{'='*60}")
print("RUNTIME RE-EMBED COMPLETE")
print(f"{'='*60}")
print(f"Model: {MODEL_PATH}")
print(f"Size: {MODEL_PATH.stat().st_size / 1e9:.2f} GB")
print(f"Version: {flx.get('version', 'unknown')}")
print(f"\nRuntime:")
print(f"  Files: {len(code_bundle)}")
print(f"  Lines: {total_lines:,}")
print(f"  Size: {compressed_size/1024:.1f} KB (compressed)")
print(f"\nFixes applied:")
print("  ✓ Absolute imports tried first (for embedded execution)")
print("  ✓ Fallback to relative imports (for direct execution)")
print(f"\n{'='*60}")

# Cleanup
del flx
gc.collect()
print("\n✓ Memory cleaned up")